In [91]:
import numpy as np

In [92]:
def create_training_data():
    data = [
        ['Sunny', 'Hot', 'High', 'Weak', 'No'],
        ['Sunny', 'Hot', 'High', 'Strong', 'No'],
        ['Overcast', 'Hot', 'High', 'Weak', 'Yes'],
        ['Rain', 'Mild', 'High', 'Weak', 'Yes'],
        ['Rain', 'Cool', 'Normal', 'Weak', 'Yes'],
        ['Rain', 'Cool', 'Normal', 'Strong', 'No'],
        ['Overcast', 'Cool', 'Normal', 'Strong', 'Yes'],
        ['Overcast', 'Mild', 'High', 'Weak', 'No'],
        ['Sunny', 'Cool', 'Normal', 'Weak', 'Yes'],
        ['Rain', 'Mild', 'Normal', 'Weak', 'Yes']
    ]

    return np.array(data)

In [93]:
train_data = create_training_data()
print(train_data)

[['Sunny' 'Hot' 'High' 'Weak' 'No']
 ['Sunny' 'Hot' 'High' 'Strong' 'No']
 ['Overcast' 'Hot' 'High' 'Weak' 'Yes']
 ['Rain' 'Mild' 'High' 'Weak' 'Yes']
 ['Rain' 'Cool' 'Normal' 'Weak' 'Yes']
 ['Rain' 'Cool' 'Normal' 'Strong' 'No']
 ['Overcast' 'Cool' 'Normal' 'Strong' 'Yes']
 ['Overcast' 'Mild' 'High' 'Weak' 'No']
 ['Sunny' 'Cool' 'Normal' 'Weak' 'Yes']
 ['Rain' 'Mild' 'Normal' 'Weak' 'Yes']]


In [94]:
def compute_prior_probabilities(train_data):

    class_names = ['No', 'Yes']
    total_samples = len(train_data)
    prior_probs = np.zeros(len(class_names))

    for i, class_name in enumerate(class_names):
        occurances = np.sum(train_data[:, -1] == class_name)
        prior_probs[i] = occurances / total_samples

    return prior_probs

prior_probablity = compute_prior_probabilities(train_data)
print("P(“Play Tennis” = No)", prior_probablity[0])
print("P(“Play Tennis” = Yes)", prior_probablity[1])

P(“Play Tennis” = No) 0.4
P(“Play Tennis” = Yes) 0.6


In [95]:
def compute_conditional_probabilities(train_data):

    class_names = ['No', 'Yes']
    n_features = train_data.shape[1] - 1 
    conditional_probs = []
    feature_values = []

    for feature_idx in range(n_features):
        unique_values = np.unique(train_data[:, feature_idx])
        feature_values.append(unique_values)

        feature_cond_probs = np.zeros((len(class_names), len(unique_values)))

        for class_idx, class_name in enumerate(class_names):
            class_samples = train_data[train_data[:, -1] == class_name]

            for value_idx, value in enumerate(unique_values):
                occurances = np.sum(class_samples[:, feature_idx] == value)
                feature_cond_probs[class_idx, value_idx] = occurances / len(class_samples)

        conditional_probs.append(feature_cond_probs)

    return conditional_probs, feature_values

In [96]:
_, feature_values  = compute_conditional_probabilities(train_data)
print("x1 = ",feature_values[0])
print("x2 = ",feature_values[1])
print("x3 = ",feature_values[2])
print("x4 = ",feature_values[3])

x1 =  ['Overcast' 'Rain' 'Sunny']
x2 =  ['Cool' 'Hot' 'Mild']
x3 =  ['High' 'Normal']
x4 =  ['Strong' 'Weak']


In [97]:
def get_feature_index(feature_value, feature_values):

    return np.where(feature_values == feature_value)[0][0]


In [98]:
_, feature_values = compute_conditional_probabilities(train_data)
outlook = feature_values[0]
i1 = get_feature_index("Overcast", outlook)
i2 = get_feature_index("Rain", outlook)
i3 = get_feature_index("Sunny", outlook)

print(i1, i2, i3)

0 1 2


In [99]:
def train_naive_bayes(train_data):

    prior_probabilities = compute_prior_probabilities(train_data)

    conditional_probabilities, feature_names = compute_conditional_probabilities(train_data)

    return prior_probabilities, conditional_probabilities, feature_names

In [100]:
prior_probs, conditional_probs, feature_names = train_naive_bayes(train_data)

In [ ]:
def predict_tennis(X, prior_probabilities, conditional_probabilities, feature_names):

    class_names = ['No', 'Yes']

    feature_indices = []
    for i, feature_value in enumerate(X):
        feature_indices.append(get_feature_index(feature_value, feature_names[i]))

    class_probabilities = []

    for class_idx in range(len(class_names)):
        probability = prior_probabilities[class_idx]

        for feature_idx, value_idx in enumerate(feature_indices):
            probability *= conditional_probabilities[feature_idx][class_idx, value_idx]

        class_probabilities.append(probability)

    total_prob = sum(class_probabilities)
    if total_prob > 0:
        normalized_probs = [p / total_prob for p in class_probabilities]
    else:
        normalized_probs = [0.5, 0.5]  

    predicted_class_idx = np.argmax(class_probabilities)
    prediction = class_names[predicted_class_idx]

    prob_dict = {
        'No': round(normalized_probs[0].item(), 2),
        'Yes': round(normalized_probs[1].item(), 2)
    }

    return prediction, prob_dict

In [102]:
X = ['Sunny','Cool', 'High', 'Strong']
prior_probs, conditional_probs, feature_names = train_naive_bayes(train_data)
prediction, prob_dict = predict_tennis(
    X, prior_probs, conditional_probs, feature_names
)
if prediction:
    print("Ad should go!")
else:
    print("Ad should not go!")
prediction, prob_dict

[np.float64(0.018750000000000003), np.float64(0.002777777777777777)]
Ad should go!


('No', {'No': 0.87, 'Yes': 0.13})